# ExoHunter — Exploratory Notebook

**A guided tour of the exoplanet transit detection pipeline**

This notebook walks through the complete ExoHunter pipeline on
**TOI-700** — a nearby M-dwarf star with three confirmed transiting
planets, including one in the habitable zone.

### What you'll learn

1. How to download TESS light curves from NASA's MAST archive
2. How preprocessing cleans and detrends the data
3. How the BLS algorithm finds periodic transit signals
4. How phase-folding reveals the transit shape
5. How iterative subtraction discovers multiple planets
6. How cross-matching classifies candidates against known catalogs

### Prerequisites

```bash
pip install -e ".[dev]"
```

### Target: TOI-700 (TIC 150428135)

| Planet | Period (d) | Depth (ppm) | Radius (R⊕) | Notes |
|--------|-----------|-------------|-------------|-------|
| **b** | 9.977 | 580 | 1.01 | Rocky, Earth-sized |
| **c** | 16.051 | 780 | 2.63 | Sub-Neptune |
| **d** | 37.426 | 810 | 1.14 | **Habitable zone!** |

In [ ]:
# Standard imports
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# ExoHunter imports
from exohunter import config
from exohunter.ingestion.downloader import download_lightcurve
from exohunter.preprocessing.pipeline import preprocess_single
from exohunter.preprocessing.clean import remove_nans, remove_outliers
from exohunter.preprocessing.normalize import normalize_lightcurve
from exohunter.preprocessing.detrend import flatten_lightcurve
from exohunter.detection.bls import run_bls_lightkurve, run_iterative_bls, TransitCandidate
from exohunter.detection.model import transit_model, phase_fold, bin_phase_curve
from exohunter.detection.validator import validate_candidate
from exohunter.catalog.crossmatch import crossmatch_candidate, MatchClass
from exohunter.catalog.candidates import CandidateCatalog, compute_score

print("All imports loaded successfully.")

---

## 1. What is a transit?

When a planet passes in front of its host star, it blocks a tiny fraction of the star's light. The animation below shows this geometry — a dark planet crossing a bright star — with the resulting light curve dip drawn in sync.

In [ ]:
# Animation: planet transiting a star with synchronized light curve

n_frames = 60
planet_x_positions = np.linspace(-1.8, 1.8, n_frames)
star_radius = 0.8
planet_radius = 0.12  # Rp/Rs ~ 0.15

# Compute flux at each frame
def compute_transit_flux(planet_x, star_r, planet_r):
    """Simple geometric transit: fraction of star blocked by planet."""
    d = abs(planet_x)
    if d >= star_r + planet_r:
        return 1.0  # no overlap
    elif d <= star_r - planet_r:
        return 1.0 - (planet_r / star_r) ** 2  # full overlap
    else:
        # Partial overlap (simplified)
        overlap = (planet_r ** 2) * np.arccos((d**2 + planet_r**2 - star_r**2) / (2 * d * planet_r)) + \
                  (star_r ** 2) * np.arccos((d**2 + star_r**2 - planet_r**2) / (2 * d * star_r)) - \
                  0.5 * np.sqrt((-d+planet_r+star_r)*(d+planet_r-star_r)*(d-planet_r+star_r)*(d+planet_r+star_r))
        return 1.0 - overlap / (np.pi * star_r ** 2)

flux_values = [compute_transit_flux(x, star_radius, planet_radius) for x in planet_x_positions]

# Build star circle
theta = np.linspace(0, 2 * np.pi, 100)
star_x = star_radius * np.cos(theta)
star_y = star_radius * np.sin(theta)

# Build frames
frames = []
for i in range(n_frames):
    planet_circle_x = planet_x_positions[i] + planet_radius * np.cos(theta)
    planet_circle_y = planet_radius * np.sin(theta)

    frames.append(go.Frame(
        data=[
            # Star (subplot 1)
            go.Scatter(x=star_x, y=star_y, fill="toself",
                       fillcolor="gold", line=dict(color="orange", width=2),
                       showlegend=False, xaxis="x", yaxis="y"),
            # Planet (subplot 1)
            go.Scatter(x=planet_circle_x, y=planet_circle_y, fill="toself",
                       fillcolor="#1a1a2e", line=dict(color="#333", width=1),
                       showlegend=False, xaxis="x", yaxis="y"),
            # Light curve trace (subplot 2)
            go.Scatter(x=list(planet_x_positions[:i+1]), y=list(flux_values[:i+1]),
                       mode="lines", line=dict(color="deepskyblue", width=3),
                       showlegend=False, xaxis="x2", yaxis="y2"),
            # Current point marker
            go.Scatter(x=[planet_x_positions[i]], y=[flux_values[i]],
                       mode="markers", marker=dict(color="red", size=10),
                       showlegend=False, xaxis="x2", yaxis="y2"),
        ],
        name=str(i),
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=500,
        width=900,
        title="Transit Geometry — Planet Crossing a Star",
        xaxis=dict(domain=[0, 0.45], range=[-2, 2], scaleanchor="y",
                   showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(range=[-1.5, 1.5], showgrid=False, zeroline=False,
                   showticklabels=False, title=""),
        xaxis2=dict(domain=[0.55, 1.0], range=[-1.9, 1.9], title="Planet position"),
        yaxis2=dict(anchor="x2", range=[0.975, 1.005], title="Normalized Flux"),
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            y=0,
            x=0.5,
            xanchor="center",
            buttons=[
                dict(label="▶ Play",
                     method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True},
                                  "fromcurrent": True}]),
                dict(label="⏸ Pause",
                     method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False},
                                    "mode": "immediate"}]),
            ],
        )],
    ),
)

fig.show()

---

## 2. Download TESS data for TOI-700

We use ExoHunter's `download_lightcurve()` function, which:
1. Checks the local FITS cache first
2. On a cache miss, queries MAST via lightkurve
3. Downloads and stitches all available sectors
4. Caches the result for future runs

In [ ]:
TIC_ID = "TIC 150428135"

light_curve = download_lightcurve(TIC_ID)

if light_curve is None:
    # Fallback: generate synthetic data for offline use
    print("Could not download — generating synthetic TOI-700 data for offline use")
    from tests.conftest import make_multi_planet_lc
    light_curve = make_multi_planet_lc(
        planets=[
            {"period": 9.977,  "epoch": 2.5, "depth": 0.00058, "duration": 0.095},
            {"period": 16.051, "epoch": 4.1, "depth": 0.00078, "duration": 0.125},
            {"period": 37.426, "epoch": 8.7, "depth": 0.00081, "duration": 0.148},
        ],
        noise=0.0003, n_points=25000, baseline_days=351.0,
    )

time_raw = np.array(light_curve.time.value, dtype=np.float64)
flux_raw = np.array(light_curve.flux.value, dtype=np.float64)

print(f"Light curve: {len(time_raw)} cadences")
print(f"Time span: {time_raw[-1] - time_raw[0]:.1f} days")
print(f"Median flux: {np.median(flux_raw):.4f}")

In [ ]:
# Plot the raw light curve
fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=time_raw, y=flux_raw, mode="markers",
    marker=dict(size=1.5, color="deepskyblue", opacity=0.4),
    name="Raw flux",
))
fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title=f"Raw Light Curve — {TIC_ID} (TOI-700)",
    xaxis_title="Time (BTJD)", yaxis_title="Flux",
    height=400, xaxis_rangeslider_visible=True,
)
fig.show()

---

## 3. Preprocessing — step by step

The pipeline applies four cleaning steps. Let's see each one's effect.

In [ ]:
# Run the full preprocessing pipeline
processed = preprocess_single(light_curve, tic_id=TIC_ID)
time_proc = processed.time
flux_proc = processed.flux

print(f"Before preprocessing: {len(time_raw)} cadences")
print(f"After preprocessing:  {len(time_proc)} cadences")
print(f"CDPP noise estimate:  {processed.cdpp:.1f} ppm")

# Compare raw vs processed
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["Raw Light Curve", "After Preprocessing"])

fig.add_trace(go.Scattergl(
    x=time_raw, y=flux_raw, mode="markers",
    marker=dict(size=1, color="gray", opacity=0.3), name="Raw",
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=time_proc, y=flux_proc, mode="markers",
    marker=dict(size=1.5, color="deepskyblue", opacity=0.5), name="Processed",
), row=2, col=1)

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)", height=500,
    title="Effect of Preprocessing Pipeline",
    showlegend=False,
)
fig.update_yaxes(title_text="Flux", row=1)
fig.update_yaxes(title_text="Normalized Flux", row=2)
fig.update_xaxes(title_text="Time (BTJD)", row=2)
fig.show()

---

## 4. BLS Transit Search

The **Box Least Squares** algorithm searches for periodic box-shaped dips.
It tests thousands of trial periods and finds the one where the data best
matches a transit model.

In [ ]:
# Run BLS on the preprocessed data
lc_for_bls = processed.to_lightcurve()

# Search a wide period range to catch TOI-700 d (P=37.4 d)
bls_period_grid = np.linspace(0.5, 45.0, 15000)
periodogram = lc_for_bls.to_periodogram(
    method="bls", period=bls_period_grid, frequency_factor=500,
)

best_period = float(periodogram.period_at_max_power.value)
best_depth = float(periodogram.depth_at_max_power.value)
best_t0 = float(periodogram.transit_time_at_max_power.value)
best_dur = float(periodogram.duration_at_max_power.value)

print(f"Strongest signal: P = {best_period:.4f} days")
print(f"Transit depth:    {best_depth * 100:.4f}%")
print(f"Duration:         {best_dur * 24:.2f} hours")

# Plot the BLS periodogram
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=periodogram.period.value, y=periodogram.power.value,
    mode="lines", line=dict(color="deepskyblue", width=1),
))

# Mark the peak and known planet periods
for p, name, color in [(best_period, "BLS peak", "red"),
                         (9.977, "TOI-700 b", "limegreen"),
                         (16.051, "TOI-700 c", "gold"),
                         (37.426, "TOI-700 d", "#00ff88")]:
    fig.add_vline(x=p, line=dict(color=color, width=1.5, dash="dash"),
                  annotation_text=f"{name} ({p:.1f}d)",
                  annotation_font_color=color, annotation_font_size=10)

fig.update_layout(
    template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    title="BLS Periodogram — Where Are the Planets?",
    xaxis_title="Period (days)", yaxis_title="BLS Power",
    height=400,
)
fig.show()

---

## 5. Phase-fold sweep animation — *why BLS works*

This is the key insight: when we fold the light curve at the **wrong** period, the transit dips land at random phases and average out to noise. When we fold at the **correct** period, all the dips line up and a clean transit shape emerges.

Press **▶ Play** and watch the data coalesce into a transit dip as the trial period approaches 9.977 days (TOI-700 b).

In [ ]:
# Phase-fold sweep animation
# We sweep through trial periods from 2 d to 14 d.
# At P ≈ 9.977 d (TOI-700 b), the transit dip appears.

trial_periods = np.concatenate([
    np.linspace(2.0, 9.0, 15),        # approach
    np.linspace(9.0, 10.5, 20),       # fine sweep around the true period
    np.linspace(10.5, 14.0, 10),      # overshoot
])

# Subsample data for animation performance
rng = np.random.default_rng(42)
n_show = min(5000, len(time_proc))
idx = rng.choice(len(time_proc), n_show, replace=False)
idx.sort()
t_sub = time_proc[idx]
f_sub = flux_proc[idx]

frames = []
for trial_p in trial_periods:
    phase = ((t_sub - best_t0) % trial_p) / trial_p
    phase = np.where(phase > 0.5, phase - 1.0, phase)

    # Bin for cleaner visualization
    bin_edges = np.linspace(-0.5, 0.5, 101)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_means = np.full(100, np.nan)
    for i in range(100):
        mask = (phase >= bin_edges[i]) & (phase < bin_edges[i + 1])
        if np.sum(mask) >= 3:
            bin_means[i] = np.mean(f_sub[mask])

    valid = ~np.isnan(bin_means)

    frames.append(go.Frame(
        data=[
            go.Scattergl(x=phase, y=f_sub, mode="markers",
                         marker=dict(size=1.5, color="rgba(100,149,237,0.15)"),
                         showlegend=False),
            go.Scatter(x=bin_centers[valid], y=bin_means[valid],
                       mode="markers+lines",
                       marker=dict(size=4, color="deepskyblue"),
                       line=dict(color="deepskyblue", width=1.5),
                       showlegend=False),
        ],
        name=f"P={trial_p:.3f}",
        layout=go.Layout(
            title=f"Phase-Fold Sweep — Trial Period = {trial_p:.3f} d"
                  + (" ← TOI-700 b!" if abs(trial_p - 9.977) < 0.15 else ""),
        ),
    ))

# Slider steps
sliders = [dict(
    active=0,
    steps=[dict(
        method="animate",
        args=[[f"P={p:.3f}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
        label=f"{p:.1f}",
    ) for p in trial_periods],
    currentvalue=dict(prefix="Period: ", suffix=" d"),
    pad=dict(t=50),
)]

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        title=f"Phase-Fold Sweep — Trial Period = {trial_periods[0]:.3f} d",
        xaxis=dict(title="Orbital Phase", range=[-0.5, 0.5]),
        yaxis=dict(title="Normalized Flux"),
        height=500,
        updatemenus=[dict(
            type="buttons", showactive=False, y=0, x=0.5, xanchor="center",
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, {"frame": {"duration": 200, "redraw": True},
                                  "fromcurrent": True}]),
                dict(label="⏸ Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
            ],
        )],
        sliders=sliders,
    ),
)
fig.show()

---

## 6. Validation and Cross-matching

Every detection is tested against 6 astrophysical criteria and then
compared to the ExoFOP-TESS TOI catalog.

In [ ]:
# Build a TransitCandidate from the BLS result
candidate = TransitCandidate(
    tic_id=TIC_ID, period=best_period, epoch=best_t0,
    duration=best_dur, depth=best_depth, snr=0.0, bls_power=0.0,
    n_transits=max(1, int((time_proc[-1] - time_proc[0]) / best_period)),
)

# Validate
validation = validate_candidate(candidate, time=time_proc, flux=flux_proc)

print("Validation Results:")
print(f"  Overall: {'VALID' if validation.is_valid else 'REJECTED'}")
for test_name, passed in validation.tests.items():
    emoji = "✓" if passed else "✗"
    print(f"  {emoji} {test_name}: {'passed' if passed else 'FAILED'}")
if validation.flags:
    print(f"\n  Flags: {'; '.join(validation.flags)}")

# Cross-match
xmatch = crossmatch_candidate(candidate)
print(f"\nCross-match: {xmatch.match_class.value}")
if xmatch.catalog_name:
    print(f"  Catalog match: {xmatch.catalog_name} (ΔP = {xmatch.period_difference:.4f} d)")

# Score
score = compute_score(candidate, validation)
print(f"\nPriority score: {score:.1f}")

---

## 7. Multi-planet search — finding all 3 planets

TOI-700 has 3 planets. BLS found the strongest one above. Now we use
**iterative subtraction**: find a planet, subtract its transit model,
re-run BLS on the residuals to find the next.

In [ ]:
# Run iterative BLS to find multiple planets
candidates = run_iterative_bls(
    lc_for_bls, tic_id=TIC_ID,
    min_period=0.5, max_period=45.0, num_periods=15000,
    max_planets=5, min_snr=0.0,  # min_snr=0 for synthetic data compatibility
)

print(f"Found {len(candidates)} candidate(s):\n")
known = {"b": 9.977, "c": 16.051, "d": 37.426}

for i, c in enumerate(candidates):
    letter = chr(ord("b") + i)
    match = ""
    for k_name, k_period in known.items():
        if abs(c.period - k_period) < 1.0:
            match = f"  ← matches TOI-700 {k_name}!"
    print(f"  Planet {letter}: P = {c.period:.4f} d, depth = {c.depth*100:.4f}%{match}")

### Planet subtraction animation

Watch each planet being removed from the light curve, revealing the next hidden signal.

In [ ]:
# Planet subtraction sequence — show each planet being removed
# Then phase-fold the residual at the next planet's period

n_planets = min(len(candidates), 3)
if n_planets >= 2:
    fig = make_subplots(
        rows=n_planets, cols=2,
        subplot_titles=[
            item for c in candidates[:n_planets]
            for item in [
                f"Light Curve (after subtracting previous planets)",
                f"Phase-fold at P={c.period:.3f} d",
            ]
        ],
        horizontal_spacing=0.08, vertical_spacing=0.12,
    )

    residual_flux = flux_proc.copy()
    colors = ["#ff6b6b", "#4ecdc4", "#ffd93d", "#6bcb77", "#9b59b6"]

    for i, c in enumerate(candidates[:n_planets]):
        row = i + 1
        color = colors[i % len(colors)]
        letter = chr(ord("b") + i)

        # Left panel: residual light curve (subsampled for performance)
        sub_idx = rng.choice(len(time_proc), min(3000, len(time_proc)), replace=False)
        sub_idx.sort()

        fig.add_trace(go.Scattergl(
            x=time_proc[sub_idx], y=residual_flux[sub_idx],
            mode="markers", marker=dict(size=1.5, color="deepskyblue", opacity=0.3),
            showlegend=False,
        ), row=row, col=1)

        # Right panel: phase-fold at this planet's period
        phase, flux_folded = phase_fold(time_proc, residual_flux, c.period, c.epoch)
        centers, means, stds = bin_phase_curve(phase, flux_folded, n_bins=150)

        fig.add_trace(go.Scatter(
            x=centers, y=means, mode="markers+lines",
            marker=dict(size=3, color=color),
            line=dict(color=color, width=1.5),
            name=f"Planet {letter} (P={c.period:.3f} d)",
        ), row=row, col=2)

        # Subtract this planet's transit for the next iteration
        model = transit_model(time_proc, c.period, c.epoch, c.duration, c.depth)
        residual_flux = residual_flux - (model - 1.0)

    fig.update_layout(
        template="plotly_dark", paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=300 * n_planets,
        title="Iterative Planet Subtraction — Peeling Back the Layers",
    )
    # Zoom phase plots to transit region
    for i in range(n_planets):
        fig.update_xaxes(range=[-0.15, 0.15], row=i+1, col=2)
    fig.show()
else:
    print("Need at least 2 candidates for the subtraction animation.")

---

## 8. Summary

In this notebook we:

1. **Animated** transit geometry to understand what a transit signal looks like
2. **Downloaded** real TESS data for TOI-700 from NASA's MAST archive
3. **Preprocessed** the light curve (cleaning, normalization, detrending)
4. **Ran BLS** to detect periodic transit signals and visualized the periodogram
5. **Animated** the phase-fold sweep to see *why* the correct period produces a signal
6. **Validated** the candidate against 6 astrophysical criteria
7. **Cross-matched** against the ExoFOP-TESS TOI catalog
8. **Found multiple planets** using iterative subtraction and visualized the process

### Next steps

- Open `02_student_exercises.ipynb` to practice these techniques on your own
- Run `python scripts/run_batch.py --sector 56 --multi-planet` to search an entire sector
- Try the interactive dashboard: `python scripts/run_dashboard.py`

### Key takeaway

> A transit is just a tiny periodic dip in brightness — less than 0.1% for
> an Earth-sized planet. Finding it requires downloading data from space
> telescopes, carefully removing noise, and searching through thousands of
> trial periods. When the right period is found, the signal emerges from
> the noise like magic. That's what ExoHunter does.